# DP-600 · Session 2 · Notebook 1 of 3 — Lakehouse with PySpark
**Transforming and querying data with SQL, KQL and DAX** · Ladislau André

This notebook **creates all the demo data** and then shapes it with PySpark, following the deck:

| Section | Slide | What you show |
|---|---|---|
| 0 | — | Setup (schemas `bronze`, `silver`, `gold`) |
| 1 | — | Generate dirty raw data (customers, products, stores, sales 2025 + 2026) |
| 2 | 7 | Clean: duplicates, nulls, wrong types, unwanted rows |
| 3 | 8 | Keep the latest row per customer (window function) |
| 4 | 6 | Reshape: denormalize, merge (append), join types, aggregate |
| 5 | 4 | Add columns with `withColumn` |
| 6 | 5 | Build the star schema with a Type 2 customer dimension |
| 7 | 5 | Apply Type 1 and Type 2 changes with Delta `MERGE` |
| 8 | — | Check the result |

**Before you run:** attach the lakehouse **`LH_DP600`** (schemas enabled) as the default lakehouse, then *Run all* once before the session (about 2–3 minutes). During the talk, re-run single cells.

> Why Spark here? Generating data, cleaning files and writing Delta tables in a lakehouse is Spark's job — the lakehouse SQL analytics endpoint is **read-only** (slide 3).

## 0 · Setup

In [ ]:
from pyspark.sql import functions as F, Window as W
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType
import random, datetime as dt

SEED = 42            # same seed = same data every time
rnd = random.Random(SEED)

for schema in ["bronze", "silver", "gold"]:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {schema}")

def save(df, table):
    """Overwrite a Delta table in the attached lakehouse."""
    (df.write.mode("overwrite").option("overwriteSchema", "true")
       .format("delta").saveAsTable(table))
    print(f"✔ {table}: {spark.table(table).count():,} rows")

## 1 · Generate the raw (bronze) data
Everything below is synthetic and seeded. The raw tables are **deliberately dirty** so the cleaning demos have something to fix:

* `bronze.customers_raw` — several versions per customer (people move), exact duplicates, missing cities, messy names and phones, dates stored as text
* `bronze.categories`, `bronze.subcategories`, `bronze.products` — a **normalized** product hierarchy (to denormalize later)
* `bronze.stores`
* `bronze.sales_2025` and `bronze.sales_2026` — same columns, **all text** (like a CSV landing zone), with duplicates, `TEST` rows, `N/A` amounts, missing quantities and orders from an unknown customer

In [ ]:
# --- Product hierarchy (normalized: category -> subcategory -> product) ---
categories = [(1, "Bikes"), (2, "Components"), (3, "Clothing"), (4, "Accessories"), (5, "Gift Cards")]
subcategories = [
    (10, 1, "Road Bikes"), (11, 1, "Mountain Bikes"), (12, 1, "Touring Bikes"),
    (20, 2, "Wheels"), (21, 2, "Brakes"), (22, 2, "Chains"),
    (30, 3, "Jerseys"), (31, 3, "Gloves"), (32, 3, "Caps"),
    (40, 4, "Helmets"), (41, 4, "Bottles"), (42, 4, "Lights"),
    (50, 5, "Gift Cards"),
]
price_range = {1: (650, 3200), 2: (20, 420), 3: (12, 95), 4: (6, 70)}
colors = ["Black", "Red", "Blue", "Silver", "White", None]   # None = missing colour

products = []
pid = 1001
for sub_id, cat_id, sub in subcategories:
    models = ["10", "25"] if cat_id == 5 else ["100", "200", "Pro"]
    for m in models:
        price = float(m) if cat_id == 5 else round(rnd.uniform(*price_range[cat_id]), 2)
        color = None if cat_id == 5 else rnd.choice(colors)
        products.append((pid, f"{sub.rstrip('s')} {m}", sub_id, price, color))
        pid += 1

stores = [
    ("S01", "Lisbon Flagship", "Lisbon", "Portugal"),
    ("S02", "Porto Riverside", "Porto", "Portugal"),
    ("S03", "Madrid Centro", "Madrid", "Spain"),
    ("S04", "Barcelona Port", "Barcelona", "Spain"),
    ("S05", "London Soho", "London", "United Kingdom"),
    ("S06", "Manchester Arena", "Manchester", "United Kingdom"),
    ("S07", "São Paulo Paulista", "São Paulo", "Brazil"),
    ("S08", "Online Store", "Online", "Global"),
]

save(spark.createDataFrame(categories, "CategoryID int, Category string"), "bronze.categories")
save(spark.createDataFrame(subcategories, "SubcategoryID int, CategoryID int, Subcategory string"), "bronze.subcategories")
save(spark.createDataFrame(products, "ProductID int, ProductName string, SubcategoryID int, ListPrice double, Color string"), "bronze.products")
save(spark.createDataFrame(stores, "StoreID string, StoreName string, City string, Country string"), "bronze.stores")

In [ ]:
# --- Customers: 300 people, some move once or twice (versions with ModifiedDate) ---
first = ["Ana", "João", "Maria", "Pedro", "Sofia", "Lucas", "Inês", "Tiago", "Laura", "Diego",
         "Carmen", "Pablo", "Emma", "Oliver", "Amelia", "Harry", "Beatriz", "Rafael", "Julia", "Mateus"]
last  = ["Costa", "Silva", "Santos", "Ferreira", "García", "López", "Martín", "Smith", "Jones",
         "Taylor", "Brown", "Oliveira", "Souza", "Pereira", "Rodríguez", "Fernández"]
cities = {
    "Portugal": ["Lisbon", "Porto", "Braga", "Coimbra"],
    "Spain": ["Madrid", "Barcelona", "Seville", "Bilbao"],
    "United Kingdom": ["London", "Manchester", "Leeds", "Bristol"],
    "Brazil": ["São Paulo", "Rio de Janeiro", "Curitiba"],
}
prefix = {"Portugal": "+351", "Spain": "+34", "United Kingdom": "+44", "Brazil": "+55"}

def rand_date(start, end):
    return start + dt.timedelta(days=rnd.randint(0, (end - start).days))

def messy_name(name):              # dirty names: random case and spaces
    r = rnd.random()
    if r < 0.08: return "  " + name.upper() + " "
    if r < 0.15: return name.lower() + "  "
    return name

def messy_phone(country):          # dirty phones: different formats, junk values
    digits = f"{rnd.randint(900000000, 999999999)}"
    r = rnd.random()
    if r < 0.05: return None
    if r < 0.08: return "unknown"
    if r < 0.40: return digits
    if r < 0.70: return f"{prefix[country]} {digits[:3]} {digits[3:6]} {digits[6:]}"
    return f"({prefix[country]}) {digits}"

customer_rows = []
for cid in range(1, 301):
    if cid == 1:   # demo customer: moves Lisbon -> Porto -> London
        name = "Ana Costa"
        versions = [("Lisbon", "Portugal", dt.date(2024, 1, 15)),
                    ("Porto", "Portugal", dt.date(2025, 6, 1)),
                    ("London", "United Kingdom", dt.date(2026, 3, 1))]
    else:
        name = f"{rnd.choice(first)} {rnd.choice(last)}"
        country = rnd.choice(list(cities))
        versions = [(rnd.choice(cities[country]), country, rand_date(dt.date(2023, 1, 1), dt.date(2024, 12, 31)))]
        r = rnd.random()
        if r < 0.30:   # moved during 2025
            versions.append((rnd.choice(cities[country]), country, rand_date(dt.date(2025, 2, 1), dt.date(2025, 12, 31))))
        if r < 0.08:   # moved again during 2026
            new_country = rnd.choice(list(cities))
            versions.append((rnd.choice(cities[new_country]), new_country, rand_date(dt.date(2026, 1, 1), dt.date(2026, 9, 30))))
    email = name.lower().replace(" ", ".") + f"{cid}@example.com"
    for city, country, modified in versions:
        city_value = None if (cid != 1 and rnd.random() < 0.05) else city      # missing city
        row = (cid, messy_name(name), city_value, country, messy_phone(country), email, modified.isoformat())
        customer_rows.append(row)
        if rnd.random() < 0.06:                                                 # exact duplicate
            customer_rows.append(row)

customers_schema = "CustomerID int, Name string, City string, Country string, Phone string, Email string, ModifiedDate string"
save(spark.createDataFrame(customer_rows, customers_schema), "bronze.customers_raw")

In [ ]:
# --- Sales: one table per year, every column as text (like a CSV landing zone) ---
cat_weight = {1: 0.12, 2: 0.20, 3: 0.28, 4: 0.395, 5: 0.005}
sub_cat = {s[0]: s[1] for s in subcategories}
prod_by_cat = {c: [p for p in products if sub_cat[p[2]] == c] for c in cat_weight}
store_ids = [s[0] for s in stores]

def sales_for_year(year, n_orders, last_day, order_start):
    rows = []
    start = dt.date(year, 1, 1)
    for i in range(n_orders):
        order_id = order_start + i
        order_date = rand_date(start, last_day)
        customer = "9999" if rnd.random() < 0.005 else str(rnd.randint(1, 300))   # 9999 = unknown customer
        store = rnd.choice(store_ids)
        status = "TEST" if rnd.random() < 0.01 else "Completed"
        for line in range(1, rnd.randint(1, 4) + 1):
            cat = rnd.choices(list(cat_weight), weights=list(cat_weight.values()))[0]
            p = rnd.choice(prod_by_cat[cat])
            qty = 1 if cat == 1 else rnd.randint(1, 2 if cat == 5 else 5)
            price = p[3]
            amount = f"{qty * price:.2f}"
            qty_txt = str(qty)
            r = rnd.random()
            if r < 0.01: amount = "N/A"        # text in a numeric column
            elif r < 0.02: qty_txt = None      # missing quantity
            row = (str(order_id), str(line), order_date.isoformat(), customer, str(p[0]), store,
                   qty_txt, f"{price:.2f}", amount, status)
            rows.append(row)
            if rnd.random() < 0.02:            # exact duplicate line
                rows.append(row)
    return rows

sales_cols = ["OrderID", "OrderLine", "OrderDate", "CustomerID", "ProductID", "StoreID",
              "Quantity", "UnitPrice", "Amount", "Status"]
sales_schema = StructType([StructField(c, StringType()) for c in sales_cols])

rows_2025 = sales_for_year(2025, 6000, dt.date(2025, 12, 31), 100000)
rows_2026 = sales_for_year(2026, 5500, dt.date(2026, 11, 10), 200000)

# Guaranteed orders for Ana Costa (customer 1) in each of her three cities -> Type 2 demo
bike = products[0]   # first road bike
for oid, d, store, target in [(190001, "2025-03-10", "S01", rows_2025), (190002, "2025-09-20", "S02", rows_2025),
                              (290001, "2026-05-05", "S05", rows_2026)]:
    target.append((str(oid), "1", d, "1", str(bike[0]), store, "1", f"{bike[3]:.2f}", f"{bike[3]:.2f}", "Completed"))

save(spark.createDataFrame(rows_2025, sales_schema), "bronze.sales_2025")
save(spark.createDataFrame(rows_2026, sales_schema), "bronze.sales_2026")

In [ ]:
# Peek at the dirty data
display(spark.table("bronze.sales_2025").filter("Amount = 'N/A' OR Quantity IS NULL OR Status = 'TEST'").limit(10))
display(spark.table("bronze.customers_raw").filter("CustomerID = 1"))

## 2 · Clean: duplicates, nulls, wrong types, unwanted rows  *(slide 7)*

| Problem | PySpark |
|---|---|
| Duplicates | `dropDuplicates()` |
| Missing or null | `fillna()`, `dropna()`, `coalesce()` |
| Wrong data type | `cast()` / `try_cast` |
| Unwanted rows | `filter()` |

First the two yearly files are **appended** (slide 6: merge = stack rows with the same columns).

In [ ]:
# Merge (append): stack 2026 under 2025 — same columns, so unionByName
sales_raw = spark.table("bronze.sales_2025").unionByName(spark.table("bronze.sales_2026"))

steps = {"1. raw (2025 + 2026)": sales_raw.count()}

s = sales_raw.dropDuplicates()                                    # duplicates
steps["2. dropDuplicates()"] = s.count()

s = s.filter(F.col("Status") != "TEST")                           # unwanted rows
steps["3. filter(Status != 'TEST')"] = s.count()

s = (s.withColumn("OrderID",    F.col("OrderID").cast("int"))     # wrong data types
      .withColumn("OrderLine",  F.col("OrderLine").cast("int"))
      .withColumn("OrderDate",  F.to_date("OrderDate", "yyyy-MM-dd"))
      .withColumn("CustomerID", F.col("CustomerID").cast("int"))
      .withColumn("ProductID",  F.col("ProductID").cast("int"))
      .withColumn("Quantity",   F.col("Quantity").cast("int"))
      .withColumn("UnitPrice",  F.col("UnitPrice").cast("decimal(18,2)"))
      .withColumn("AmountRaw",  F.expr("try_cast(Amount AS decimal(18,2))")))   # 'N/A' -> NULL, no failure

print("Rows where the amount could not be converted:", s.filter("AmountRaw IS NULL").count())
print("Rows with a missing quantity:", s.filter("Quantity IS NULL").count())

s = (s.fillna({"Quantity": 1})                                    # nulls: default quantity
      .withColumn("Amount", F.coalesce("AmountRaw",               # nulls: recompute the amount
                                       (F.col("Quantity") * F.col("UnitPrice")).cast("decimal(18,2)")))
      .drop("AmountRaw", "Status"))
steps["4. after cast + fillna + coalesce"] = s.count()

for k, v in steps.items():
    print(f"{k:<38}{v:>8,}")

save(s, "silver.sales")

In [ ]:
# Customers: exact duplicates, messy names, missing cities, phone formats, text dates
c = spark.table("bronze.customers_raw")
print("raw rows:", c.count(), "| after dropDuplicates:", c.dropDuplicates().count())

customers_clean = (c.dropDuplicates()
    .withColumn("Name", F.initcap(F.trim(F.regexp_replace("Name", r"\s+", " "))))
    .fillna({"City": "Unknown"})
    .withColumn("Phone", F.regexp_replace("Phone", r"[^0-9+]", ""))
    .withColumn("Phone", F.when(F.length("Phone") >= 9, F.col("Phone")))   # 'unknown' or empty -> NULL
    .withColumn("ModifiedDate", F.to_date("ModifiedDate")))

save(customers_clean, "silver.customers")
display(customers_clean.orderBy("CustomerID", "ModifiedDate").limit(10))

## 3 · Keep the latest row per customer  *(slide 8, in PySpark)*
Same logic as the SQL on the slide: partition by the business key, order newest first, keep row 1.

In [ ]:
w = W.partitionBy("CustomerID").orderBy(F.col("ModifiedDate").desc())

latest = (spark.table("silver.customers")
          .withColumn("rn", F.row_number().over(w))
          .filter("rn = 1")
          .drop("rn"))

print("versions:", spark.table("silver.customers").count(), "→ customers:", latest.count())
display(latest.filter("CustomerID = 1"))

## 4 · Reshape: denormalize, join, aggregate  *(slide 6)*

In [ ]:
# Denormalize: collapse Product -> Subcategory -> Category into one table
product_flat = (spark.table("bronze.products").alias("p")
    .join(spark.table("bronze.subcategories").alias("s"), "SubcategoryID")
    .join(spark.table("bronze.categories").alias("c"), "CategoryID")
    .select("ProductID", "ProductName", "Subcategory", "Category", "ListPrice",
            F.coalesce("Color", F.lit("N/A")).alias("Color")))
display(product_flat.orderBy("ProductID"))

In [ ]:
# Join types: orders carry customer 9999, which does not exist; some customers never bought
orders = spark.table("silver.sales").select("OrderID", "CustomerID").distinct()

print(f"{'inner (matches only)':<32}{orders.join(latest, 'CustomerID', 'inner').count():>8,}")
print(f"{'left  (every order)':<32}{orders.join(latest, 'CustomerID', 'left').count():>8,}")
print(f"{'full  (everything)':<32}{orders.join(latest, 'CustomerID', 'full').count():>8,}")
print(f"{'left_anti (orders w/o customer)':<32}{orders.join(latest, 'CustomerID', 'left_anti').count():>8,}")

In [ ]:
# Aggregate: roll daily sales up to monthly totals per store
monthly = (spark.table("silver.sales")
    .join(spark.table("bronze.stores"), "StoreID")
    .groupBy(F.date_format("OrderDate", "yyyy-MM").alias("YearMonth"), "StoreID", "StoreName")
    .agg(F.sum("Amount").alias("Sales"), F.countDistinct("OrderID").alias("Orders"))
    .orderBy("YearMonth", "StoreID"))

save(monthly, "gold.MonthlyStoreSales")
display(monthly.limit(16))

## 5 · Add the columns the business needs  *(slide 4: `withColumn`, then write back as Delta)*

In [ ]:
product_enriched = (product_flat
    .withColumn("PriceBand", F.when(F.col("ListPrice") >= 500, "Premium")
                              .when(F.col("ListPrice") >= 50, "Mid")
                              .otherwise("Entry"))
    .withColumn("IsBike", F.col("Category") == "Bikes"))

display(product_enriched.groupBy("Category", "PriceBand").count().orderBy("Category", "PriceBand"))

## 6 · Build the star schema  *(slide 5)*

* **Dimensions** with a surrogate key: `DimDate`, `DimProduct`, `DimStore`, `DimCustomer`
* `DimCustomer` is **Type 2**: one row per version, with `StartDate`, `EndDate` and `IsCurrent`, plus an *Unknown* member (`-1`) for orders whose customer doesn't exist
* **Fact** `FactSales`: one row per order line, numeric measures and a foreign key to each dimension. The customer key is looked up **as of the order date**, so old sales keep the old address.

In [ ]:
# DimDate
dates = spark.sql("SELECT explode(sequence(to_date('2025-01-01'), to_date('2026-12-31'), interval 1 day)) AS Date")
dim_date = (dates
    .withColumn("DateKey", F.date_format("Date", "yyyyMMdd").cast("int"))
    .withColumn("Year", F.year("Date"))
    .withColumn("Quarter", F.concat(F.lit("Q"), F.quarter("Date")))
    .withColumn("MonthNumber", F.month("Date"))
    .withColumn("MonthName", F.date_format("Date", "MMMM"))
    .withColumn("YearMonth", F.date_format("Date", "yyyy-MM"))
    .withColumn("IsWeekend", F.dayofweek("Date").isin(1, 7))
    .select("DateKey", "Date", "Year", "Quarter", "MonthNumber", "MonthName", "YearMonth", "IsWeekend"))
save(dim_date, "gold.DimDate")

# DimProduct (denormalized + enriched) and DimStore, with surrogate keys
dim_product = product_enriched.withColumn("ProductKey", F.row_number().over(W.orderBy("ProductID"))) \
    .select("ProductKey", "ProductID", "ProductName", "Subcategory", "Category", "ListPrice", "Color", "PriceBand", "IsBike")
save(dim_product, "gold.DimProduct")

dim_store = spark.table("bronze.stores").withColumn("StoreKey", F.row_number().over(W.orderBy("StoreID"))) \
    .select("StoreKey", "StoreID", "StoreName", "City", "Country")
save(dim_store, "gold.DimStore")

In [ ]:
# DimCustomer — Type 2 slowly changing dimension built from the version history
v = W.partitionBy("CustomerID").orderBy("ModifiedDate")
next_change = F.lead("ModifiedDate").over(v)

dim_customer = (spark.table("silver.customers")
    .withColumn("StartDate", F.col("ModifiedDate"))
    .withColumn("EndDate", F.coalesce(F.date_sub(next_change, 1), F.to_date(F.lit("9999-12-31"))))
    .withColumn("IsCurrent", next_change.isNull())
    .withColumn("CustomerKey", F.row_number().over(W.orderBy("CustomerID", "StartDate")))
    .select("CustomerKey", "CustomerID", "Name", "City", "Country", "Phone", "Email",
            "StartDate", "EndDate", "IsCurrent"))

unknown = spark.createDataFrame(
    [(-1, -1, "Unknown", "Unknown", "Unknown", None, None, dt.date(1900, 1, 1), dt.date(9999, 12, 31), True)],
    dim_customer.schema)

save(dim_customer.unionByName(unknown), "gold.DimCustomer")
display(spark.table("gold.DimCustomer").filter("CustomerID = 1"))

In [ ]:
# FactSales — look up every surrogate key; customer key "as of" the order date
s  = spark.table("silver.sales").alias("s")
dc = spark.table("gold.DimCustomer").alias("dc")
dp = spark.table("gold.DimProduct").alias("dp")
ds = spark.table("gold.DimStore").alias("ds")

fact_sales = (s
    .join(dc, (F.col("s.CustomerID") == F.col("dc.CustomerID")) &
              F.col("s.OrderDate").between(F.col("dc.StartDate"), F.col("dc.EndDate")), "left")
    .join(dp, F.col("s.ProductID") == F.col("dp.ProductID"), "left")
    .join(ds, F.col("s.StoreID") == F.col("ds.StoreID"), "left")
    .select(
        F.col("s.OrderID"), F.col("s.OrderLine"),
        F.date_format("s.OrderDate", "yyyyMMdd").cast("int").alias("DateKey"),
        F.col("dp.ProductKey"),
        F.coalesce(F.col("dc.CustomerKey"), F.lit(-1)).alias("CustomerKey"),   # unknown member
        F.col("ds.StoreKey"),
        F.col("s.Quantity"), F.col("s.UnitPrice"), F.col("s.Amount")))

save(fact_sales, "gold.FactSales")
print("Lines mapped to the Unknown customer:", fact_sales.filter("CustomerKey = -1").count())

In [ ]:
# Proof of Type 2: Ana Costa's sales land on the address she had on each order date
display(spark.sql("""
    SELECT c.CustomerKey, c.City, c.Country, c.StartDate, c.EndDate,
           MIN(d.Date) AS FirstOrder, MAX(d.Date) AS LastOrder,
           COUNT(*) AS Lines, SUM(f.Amount) AS Sales
    FROM gold.FactSales f
    JOIN gold.DimCustomer c ON f.CustomerKey = c.CustomerKey
    JOIN gold.DimDate d     ON f.DateKey = d.DateKey
    WHERE c.CustomerID = 1
    GROUP BY c.CustomerKey, c.City, c.Country, c.StartDate, c.EndDate
    ORDER BY c.StartDate
"""))

## 7 · Apply new changes: Type 1 vs Type 2 with Delta `MERGE`  *(slide 5)*
A change batch arrives:
* **Customer 1** (Ana) moves to Manchester → **Type 2**: close the current row, insert a new version
* **Customer 2** changes phone → **Type 1**: overwrite, no history
* **Customer 3** moves to Valencia → **Type 2**

Running this cell twice is safe: the second time nothing has changed.

In [ ]:
from delta.tables import DeltaTable

dim = DeltaTable.forName(spark, "gold.DimCustomer")
current = spark.table("gold.DimCustomer").filter("IsCurrent AND CustomerID > 0")

changes = (current.filter(F.col("CustomerID").isin(1, 2, 3))
    .select("CustomerID", "Name", "City", "Country", "Phone", "Email")
    .withColumn("City", F.when(F.col("CustomerID") == 1, "Manchester")
                         .when(F.col("CustomerID") == 3, "Valencia").otherwise(F.col("City")))
    .withColumn("Country", F.when(F.col("CustomerID") == 3, "Spain").otherwise(F.col("Country")))
    .withColumn("Phone", F.when(F.col("CustomerID") == 2, F.lit("+351910000002")).otherwise(F.col("Phone"))))
changes = spark.createDataFrame(changes.collect(), changes.schema)   # freeze the batch

display(spark.table("gold.DimCustomer").filter("CustomerID IN (1, 2, 3)").orderBy("CustomerID", "StartDate"))

In [ ]:
# TYPE 1 — overwrite the phone on every row of the customer (no history kept)
(dim.alias("d")
    .merge(changes.alias("c"), "d.CustomerID = c.CustomerID")
    .whenMatchedUpdate(condition="NOT (d.Phone <=> c.Phone)", set={"Phone": "c.Phone"})
    .execute())

# TYPE 2 — find real address changes against the current row
changed = (changes.alias("c")
    .join(current.alias("d"), "CustomerID")
    .where("c.City <> d.City OR c.Country <> d.Country")
    .select("CustomerID", "c.Name", "c.City", "c.Country", "c.Phone", "c.Email"))
changed = spark.createDataFrame(changed.collect(), changed.schema)   # freeze before the table changes

# 1) close the current version
(dim.alias("d")
    .merge(changed.alias("c"), "d.CustomerID = c.CustomerID AND d.IsCurrent = true")
    .whenMatchedUpdate(set={"EndDate": "date_sub(current_date(), 1)", "IsCurrent": "false"})
    .execute())

# 2) insert the new version with a new surrogate key
max_key = spark.table("gold.DimCustomer").agg(F.max("CustomerKey")).first()[0]
new_rows = (changed
    .withColumn("CustomerKey", F.row_number().over(W.orderBy("CustomerID")) + F.lit(max_key))
    .withColumn("StartDate", F.current_date())
    .withColumn("EndDate", F.to_date(F.lit("9999-12-31")))
    .withColumn("IsCurrent", F.lit(True))
    .select(spark.table("gold.DimCustomer").columns))
new_rows.write.format("delta").mode("append").saveAsTable("gold.DimCustomer")

print("New Type 2 versions inserted:", new_rows.count())
display(spark.table("gold.DimCustomer").filter("CustomerID IN (1, 2, 3)").orderBy("CustomerID", "StartDate"))

In [ ]:
# Delta keeps every change: the MERGEs above are in the table history
display(dim.history().select("version", "timestamp", "operation", "operationMetrics").limit(5))

## 8 · Check the result
These `gold` tables feed Notebook 2 (warehouse, T-SQL) and Notebook 3 (KQL and DAX).

In [ ]:
for t in ["gold.DimDate", "gold.DimProduct", "gold.DimStore", "gold.DimCustomer", "gold.FactSales", "gold.MonthlyStoreSales"]:
    print(f"{t:<24}{spark.table(t).count():>8,}")

# The question from slide 9, in Spark SQL on the lakehouse
display(spark.sql("""
    SELECT p.Category, SUM(s.Amount) AS Sales
    FROM gold.FactSales s
    JOIN gold.DimProduct p ON s.ProductKey = p.ProductKey
    GROUP BY p.Category
    HAVING SUM(s.Amount) > 10000
    ORDER BY Sales DESC
"""))